# Data2Vec Audio Embedding Extraction (ALS Diagnosis)

This notebook extracts utterance-level embeddings using facebook/data2vec-audio-large-960h, following the exact conventions of wav2vec2 and HuBERT notebooks.
- Traverses ../Dataset/ALI_new/All
- Builds file_list as category/subject/file.wav
- Derives subject_ids as category/subject
- Resamples each file to 16kHz
- Uses masked mean pooling
- Exports embeddings as xdata2vec_<variant>.npy plus labels.npy, file_list.npy, subject_ids.npy, label_map.npy, extraction_config.json, compute_log.json in ../Embeddings/data2vec_audio_large_mydata/

In [1]:
# 1. Import libraries and set up environment (identical to hubert/wav2vec)
import random
import sys
import platform
import numpy as np
import torch
import torchaudio
import transformers
import librosa
from pathlib import Path
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXTRACTION_ENV = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'numpy': getattr(np, '__version__', 'unknown'),
    'torch': getattr(torch, '__version__', 'unknown'),
    'torchaudio': getattr(torchaudio, '__version__', 'unknown'),
    'transformers': getattr(transformers, '__version__', 'unknown'),
    'librosa': getattr(librosa, '__version__', 'unknown'),
}

/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Dataset traversal and metadata (identical to hubert/wav2vec)
DATASET_PATH = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All")
file_list = []
s_data = []
s_sr = []
s_label = []
label_map = {}
for category in sorted(DATASET_PATH.iterdir()):
    if not category.is_dir():
        continue
    for subject in sorted(category.iterdir()):
        if not subject.is_dir():
            continue
        for wav_file in sorted(subject.glob('*.wav')):
            file_list.append(f"{category.name}/{subject.name}/{wav_file.name}")
            y = category.name
            label_map.setdefault(y, len(label_map))
            s_label.append(label_map[y])
            wav, sr = librosa.load(str(wav_file), sr=None, mono=True)
            s_data.append(wav)
            s_sr.append(sr)
subject_ids = np.array([f"{file.split('/')[0]}/{file.split('/')[1]}" for file in file_list])
labels = np.array(s_label)

In [3]:
# 3. Pre-processing: resample to 16kHz, batching, attention masks (identical to hubert/wav2vec)
def resample_to_16khz(s_data, s_sr, target_freq=16000):
    resampled = []
    for waveform, orig_freq in zip(s_data, s_sr):
        wav_tensor = torch.tensor(waveform, dtype=torch.float32)
        if wav_tensor.dim() != 1:
            wav_tensor = wav_tensor.squeeze()
        if int(orig_freq) == int(target_freq):
            resampled.append(wav_tensor)
            continue
        resampler = torchaudio.transforms.Resample(orig_freq=int(orig_freq), new_freq=int(target_freq))
        resampled.append(resampler(wav_tensor))
    return resampled

data = resample_to_16khz(s_data, s_sr)
preprocessed_data = [{'input_values': wav.squeeze()} for wav in data]

def collate_fn(batch):
    input_values_list = []
    for item in batch:
        tensor = item['input_values']
        if tensor.dim() != 1:
            tensor = tensor.squeeze()
        input_values_list.append(tensor)
    padded_inputs = pad_sequence(input_values_list, batch_first=True, padding_value=0)
    attention_masks = [torch.ones(t.size(0), dtype=torch.long) for t in input_values_list]
    padded_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    return {'input_values': padded_inputs, 'attention_mask': padded_masks}

BATCH_SIZE = 8
data_loader = DataLoader(preprocessed_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [4]:
# 4. Load Data2Vec model (frozen, eval mode)
# NOTE: Some transformers versions do not expose Data2VecAudioProcessor; we don't need it because we already provide
# 16kHz waveforms + our own padding/attention_mask (same style as wav2vec/HuBERT).
from transformers import Data2VecAudioModel
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = 'facebook/data2vec-audio-large-960h'
model = Data2VecAudioModel.from_pretrained(model_name)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(device)
print('Model loaded:', model_name, '| device:', device)

Model loaded: facebook/data2vec-audio-large-960h | device: cuda


In [5]:
# 5. Feature extraction: identical method to hubert/wav2vec (per-layer masked-mean pooling + concatenation)
import numpy as np
import torch
import time

# Define layer configurations (pattern mirrors HuBERT; indices are adapted to the model depth)
n_layers = int(getattr(model.config, 'num_hidden_layers', 0))
if n_layers <= 0:
    raise ValueError('Could not determine model.config.num_hidden_layers for Data2Vec.')

mid = n_layers // 2
stable_start = min(9, max(n_layers - 3, 0))  # prefer ~9 if available; else fall back near end
stable = [stable_start, min(stable_start + 1, n_layers - 1), min(stable_start + 2, n_layers - 1)]
middle = [max(mid - 1, 0), min(mid, n_layers - 1), min(mid + 1, n_layers - 1)]
multi_depth = [0, stable[0], middle[-1], -1]

# Note: indices refer to outputs.hidden_states indices (0..N-1). Negative indices supported here.
LAYER_CONFIGS = {
    'stability': stable,
    'acoustic': [0, 1, 2] if n_layers >= 3 else list(range(n_layers)),
    'middle': middle,
    'last3': [-3, -2, -1] if n_layers >= 3 else list(range(n_layers)),
    'multi-depth': multi_depth,
    'all': 'ALL',
}

EXTRACTION_STATS = {}

def _resolve_layers(spec, *, n_states: int) -> list[int]:
    if spec == 'ALL':
        return list(range(n_states))
    idxs: list[int] = []
    for x in spec:
        i = int(x)
        if i < 0:
            i = n_states + i
        if not (0 <= i < n_states):
            raise ValueError(f'Layer index out of range: {x} resolved to {i} with n_states={n_states}')
        idxs.append(i)
    return idxs

def extract_features(layer_sets=LAYER_CONFIGS):
    """Return dict of pooled embeddings for each layer-set (model is frozen)."""
    global EXTRACTION_STATS
    all_features = {name: [] for name in layer_sets}
    batch_times_sec = []
    batch_sizes = []
    total_utts = 0
    t_total_start = time.perf_counter()

    for batch in data_loader:
        input_values = batch['input_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bs = int(input_values.shape[0])
        total_utts += bs

        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        n_states = len(hidden_states)

        for name, spec in layer_sets.items():
            layers = _resolve_layers(spec, n_states=n_states)
            pooled_features = []
            for layer in layers:
                feature = hidden_states[layer]
                seq_len = feature.size(1)
                adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()

                masked_feature = feature * adj_mask
                valid_counts = adj_mask.sum(dim=1).clamp(min=1e-9)
                mean_pooled = masked_feature.sum(dim=1) / valid_counts
                pooled_features.append(mean_pooled)

            final_features = torch.cat(pooled_features, dim=-1).cpu().numpy()
            all_features[name].append(final_features)

        t1 = time.perf_counter()
        batch_times_sec.append(float(t1 - t0))
        batch_sizes.append(bs)

    for name in all_features:
        all_features[name] = np.concatenate(all_features[name], axis=0)

    t_total_end = time.perf_counter()
    total_sec = float(t_total_end - t_total_start)
    EXTRACTION_STATS = {
        'total_utterances': int(total_utts),
        'batch_sizes': batch_sizes,
        'batch_times_sec': batch_times_sec,
        'total_time_sec': float(total_sec),
        'mean_time_sec_per_utt': float(total_sec / max(total_utts, 1)),
        'device': str(device),
        'cuda_available': bool(torch.cuda.is_available()),
        'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

    return all_features

# Run extraction
all_features = extract_features()
print('Extracted feature sets:', {k: v.shape for k, v in all_features.items()})

Extracted feature sets: {'stability': (6926, 3072), 'acoustic': (6926, 3072), 'middle': (6926, 3072), 'last3': (6926, 3072), 'multi-depth': (6926, 4096), 'all': (6926, 25600)}


In [7]:
# 6. Save EVERYTHING next to your existing HuBERT embeddings, under a new folder "data2vec"
import json
from pathlib import Path
import numpy as np

# Pull arrays out (same naming convention you use elsewhere)
last3     = all_features["last3"]
middle3   = all_features["middle"]
acoustic  = all_features["acoustic"]
stability = all_features["stability"]
full_spec = all_features["multi-depth"]
all_layers= all_features["all"]

# Use same base embeddings folder you are already using:
# /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/Hubert
# Save Data2Vec into a sibling folder named "data2vec"
out_dir = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings") / "data2vec"
out_dir.mkdir(parents=True, exist_ok=True)

# Embeddings
np.save(out_dir / "xdata2vec_last3.npy", last3)
np.save(out_dir / "xdata2vec_middle3.npy", middle3)
np.save(out_dir / "xdata2vec_acoustic.npy", acoustic)
np.save(out_dir / "xdata2vec_stability.npy", stability)
np.save(out_dir / "xdata2vec_fullspec.npy", full_spec)
np.save(out_dir / "xdata2vec_all.npy", all_layers)

# Metadata (same convention as your new pipeline expects)
np.save(out_dir / "labels.npy", np.asarray(s_label))
np.save(out_dir / "file_list.npy", np.asarray(file_list, dtype=object))

def _subject_id_from_file_id(file_id: object) -> str:
    parts = str(file_id).split("/")
    # Expected: category/subject/filename.wav
    return "/".join(parts[:2]) if len(parts) >= 2 else parts[0]

subject_ids = np.asarray([_subject_id_from_file_id(f) for f in file_list], dtype=object)
np.save(out_dir / "subject_ids.npy", subject_ids)

# label map (if present)
LABELS = globals().get("LABELS", None)
if LABELS is None:
    LABELS = dict(globals().get("label_map", {}))
np.save(out_dir / "label_map.npy", LABELS, allow_pickle=True)

# Logs/config
try:
    (out_dir / "compute_log.json").write_text(json.dumps(globals().get("EXTRACTION_STATS", {}), indent=2))
except Exception as e:
    print(f"⚠️ Could not write compute_log.json: {e}")

config = {
    "data_path": str(globals().get("DATASET_PATH", "unknown")),
    "model_name": "facebook/data2vec-audio-large-960h",
    "model_frozen": True,
    "seed": globals().get("SEED", None),
    "pooling": "masked_mean (attention-mask aware)",
    "layer_configs": LAYER_CONFIGS,
}
(out_dir / "extraction_config.json").write_text(json.dumps(config, indent=2))

print("✅ Saved Data2Vec export to:", str(out_dir.resolve()))
print("Shapes:", {
    "last3": last3.shape,
    "middle3": middle3.shape,
    "acoustic": acoustic.shape,
    "stability": stability.shape,
    "fullspec": full_spec.shape,
    "all": all_layers.shape,
})
print("Unique subjects saved:", int(np.unique(subject_ids).size))


✅ Saved Data2Vec export to: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/data2vec
Shapes: {'last3': (6926, 3072), 'middle3': (6926, 3072), 'acoustic': (6926, 3072), 'stability': (6926, 3072), 'fullspec': (6926, 4096), 'all': (6926, 25600)}
Unique subjects saved: 63


In [ ]:
print(out_dir.resolve())


# MINSIK Data

In [ ]:
#!/usr/bin/env python3
"""
DATA2VEC EMBEDDING EXTRACTION (MINSK2020 ALS DATABASE, 2-CLASS)
==============================================================

Dataset expected layout (confirmed):
  /mnt/d/PhD/ALS_Diagnosis_Meta/Dataset/Minsk2020_ALS_database-main/
    ALS/*.wav
    HC/*.wav

Each file looks like: 008_a.wav, 008_i.wav, ...
We treat the subject id as the numeric prefix before '_' (e.g., "008"),
and keep class in the subject id to avoid collisions:
  subject_id = "<class>/<subject>"   e.g., "ALS/008", "HC/008"

Output (same convention as your other SSL exports)
--------------------------------------------------
Writes to:
  /mnt/d/PhD/ALS_Diagnosis_Meta/Embeddings/minsik/raw/Data2vec/

Files:
  - xdata2vec_last3.npy
  - xdata2vec_middle3.npy
  - xdata2vec_acoustic.npy
  - xdata2vec_stability.npy
  - xdata2vec_fullspec.npy
  - xdata2vec_all.npy
  - labels.npy
  - subject_ids.npy
  - file_list.npy
  - label_map.npy
  - compute_log.json
  - extraction_config.json

Notes
-----
- Model is frozen (no gradients).
- Waveforms are resampled to 16kHz.
- Pooling is masked-mean (attention-mask aware) per selected layer(s), then concatenated.
"""

from __future__ import annotations

import json
import platform
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torchaudio
import transformers
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import Data2VecAudioModel

# torchaudio >=2.5 may default to TorchCodec for torchaudio.load(), which requires `torchcodec`.
# To keep this script working in environments without torchcodec, we:
#   - try to set a non-torchcodec backend if available
#   - and fall back to librosa if torchaudio.load fails
try:
    torchaudio.set_audio_backend("soundfile")
except Exception:
    pass
try:
    import librosa  # type: ignore
except Exception:
    librosa = None


# ==================== Step 0: Repro / environment ====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXTRACTION_ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": getattr(np, "__version__", "unknown"),
    "torch": getattr(torch, "__version__", "unknown"),
    "torchaudio": getattr(torchaudio, "__version__", "unknown"),
    "transformers": getattr(transformers, "__version__", "unknown"),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


# ==================== Step 1: Dataset traversal (2-class) ====================
REPO_ROOT = Path("/mnt/d/PhD/ALS_Diagnosis_Meta")
DATASET_PATH = REPO_ROOT / "Dataset" / "Minsk2020_ALS_database-main"

CLASS_DIRS = ["HC", "ALS"]  # force stable mapping
LABEL_MAP = {"HC": 0, "ALS": 1}


def _iter_audio_files(root: Path) -> list[Path]:
    wavs = sorted(root.rglob("*.wav"))
    if not wavs:
        raise RuntimeError(f"No .wav files found under {root}")
    return wavs


def _subject_id_from_path(p: Path) -> str:
    # p is .../<class>/<file.wav>
    cls = p.parent.name
    stem = p.stem
    subj = stem.split("_", 1)[0]  # "008" from "008_a"
    return f"{cls}/{subj}"


@dataclass(frozen=True)
class Item:
    path: Path
    file_id: str
    subject_id: str
    label: int


items: list[Item] = []
for cls in CLASS_DIRS:
    cls_dir = DATASET_PATH / cls
    if not cls_dir.exists():
        raise RuntimeError(f"Missing class folder: {cls_dir}")
    for p in sorted(cls_dir.glob("*.wav")):
        file_id = f"{cls}/{p.name}"
        items.append(
            Item(
                path=p,
                file_id=file_id,
                subject_id=_subject_id_from_path(p),
                label=int(LABEL_MAP[cls]),
            )
        )

if not items:
    raise RuntimeError(f"No audio files found in {DATASET_PATH} with classes {CLASS_DIRS}")

file_list = np.asarray([it.file_id for it in items], dtype=object)
subject_ids = np.asarray([it.subject_id for it in items], dtype=object)
labels = np.asarray([it.label for it in items], dtype=int)

print("\nSTEP 1) Dataset loaded")
print("  root:", DATASET_PATH)
print("  n_utterances:", int(len(items)))
print("  n_subjects:", int(np.unique(subject_ids).size))
print("  class_counts:", {k: int((labels == v).sum()) for k, v in LABEL_MAP.items()})


# ==================== Step 2: Audio loading + resample to 16k ====================
TARGET_SR = 16000


class WavDataset(Dataset):
    def __init__(self, items: list[Item]):
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict:
        it = self.items[idx]
        # Prefer torchaudio.load (fast) but fall back to librosa if torchaudio requires torchcodec.
        try:
            wav, sr = torchaudio.load(str(it.path))  # (ch, n)
            if wav.ndim == 2 and wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)
            wav = wav.squeeze(0).contiguous()  # (n,)
        except Exception as e:
            if librosa is None:
                raise RuntimeError(
                    "Audio load failed via torchaudio and librosa is not available. "
                    "Install one of: torchcodec (for torchaudio default backend) or librosa/soundfile."
                ) from e
            wav_np, sr = librosa.load(str(it.path), sr=None, mono=True)
            wav = torch.tensor(wav_np, dtype=torch.float32).contiguous()

        if int(sr) != int(TARGET_SR):
            wav = torchaudio.transforms.Resample(orig_freq=int(sr), new_freq=int(TARGET_SR))(wav)

        return {"input_values": wav, "label": it.label, "subject_id": it.subject_id, "file_id": it.file_id}


def collate_fn(batch: list[dict]) -> dict:
    xs = [b["input_values"].float().squeeze() for b in batch]
    padded = pad_sequence(xs, batch_first=True, padding_value=0.0)  # (B, T)
    masks = [torch.ones(x.size(0), dtype=torch.long) for x in xs]
    padded_mask = pad_sequence(masks, batch_first=True, padding_value=0)  # (B, T)
    return {
        "input_values": padded,
        "attention_mask": padded_mask,
        "labels": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "subject_ids": [b["subject_id"] for b in batch],
        "file_ids": [b["file_id"] for b in batch],
    }


BATCH_SIZE = 8
loader = DataLoader(WavDataset(items), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print("\nSTEP 2) DataLoader ready | batch_size:", BATCH_SIZE, "| target_sr:", TARGET_SR)


# ==================== Step 3: Load Data2Vec model (frozen) ====================
model_name = "facebook/data2vec-audio-large-960h"
print("\nSTEP 3) Loading model:", model_name)
model = Data2VecAudioModel.from_pretrained(model_name)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(device)

n_layers = int(getattr(model.config, "num_hidden_layers", 0))
if n_layers <= 0:
    raise ValueError("Could not determine model.config.num_hidden_layers for Data2Vec.")

print("  num_hidden_layers:", n_layers, "| hidden_size:", int(getattr(model.config, "hidden_size", -1)))


# ==================== Step 4: Layer configs (match your naming) ====================
mid = n_layers // 2
stable_start = min(9, max(n_layers - 3, 0))
stable_layers = [stable_start, min(stable_start + 1, n_layers - 1), min(stable_start + 2, n_layers - 1)]
middle_layers = [max(mid - 1, 0), min(mid, n_layers - 1), min(mid + 1, n_layers - 1)]
multi_depth_layers = [0, stable_layers[0], middle_layers[-1], -1]

LAYER_CONFIGS: dict[str, list[int] | str] = {
    "stability": stable_layers,
    "acoustic": [0, 1, 2] if n_layers >= 3 else list(range(n_layers)),
    "middle3": middle_layers,
    "last3": [-3, -2, -1] if n_layers >= 3 else list(range(n_layers)),
    "fullspec": multi_depth_layers,
    "all": "ALL",
}


def _resolve_layers(spec: list[int] | str, *, n_states: int) -> list[int]:
    if spec == "ALL":
        return list(range(n_states))
    idxs: list[int] = []
    for x in spec:
        i = int(x)
        if i < 0:
            i = n_states + i
        if not (0 <= i < n_states):
            raise ValueError(f"Layer index out of range: {x} resolved to {i} with n_states={n_states}")
        idxs.append(i)
    return idxs


# ==================== Step 5: Feature extraction (masked-mean pooling) ====================
print("\nSTEP 5) Extracting features (frozen forward only)...")
all_features: dict[str, list[np.ndarray]] = {name: [] for name in LAYER_CONFIGS.keys()}

batch_times_sec: list[float] = []
batch_sizes: list[int] = []
t_total_start = time.perf_counter()
total_utts = 0

for bi, batch in enumerate(loader, start=1):
    input_values = batch["input_values"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    bs = int(input_values.size(0))
    total_utts += bs

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    n_states = len(hidden_states)

    for name, spec in LAYER_CONFIGS.items():
        layers = _resolve_layers(spec, n_states=n_states)
        pooled_features = []
        for layer in layers:
            feat = hidden_states[layer]  # (B, T', H)
            seq_len = feat.size(1)
            adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()
            masked = feat * adj_mask
            valid = adj_mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = masked.sum(dim=1) / valid  # (B, H)
            pooled_features.append(mean_pooled)
        final = torch.cat(pooled_features, dim=-1).cpu().numpy()
        all_features[name].append(final)

    t1 = time.perf_counter()
    batch_times_sec.append(float(t1 - t0))
    batch_sizes.append(bs)

    if bi == 1 or bi % 10 == 0 or bi == int(np.ceil(len(items) / BATCH_SIZE)):
        print(f"  batch {bi:03d} | bs={bs} | time={batch_times_sec[-1]:.3f}s")

for name in list(all_features.keys()):
    all_features[name] = [np.concatenate(all_features[name], axis=0)]

t_total_end = time.perf_counter()
total_sec = float(t_total_end - t_total_start)

shapes = {k: v[0].shape for k, v in all_features.items()}
print("\nExtracted feature sets (shapes):", shapes)

EXTRACTION_STATS = {
    "total_utterances": int(total_utts),
    "batch_sizes": batch_sizes,
    "batch_times_sec": batch_times_sec,
    "total_time_sec": float(total_sec),
    "mean_time_sec_per_utt": float(total_sec / max(total_utts, 1)),
    "device": str(device),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}


# ==================== Step 6: Save export ====================
out_dir = REPO_ROOT / "Embeddings" / "minsik" / "raw" / "Data2vec"
out_dir.mkdir(parents=True, exist_ok=True)

print("\nSTEP 6) Saving to:", out_dir)

np.save(out_dir / "xdata2vec_last3.npy", all_features["last3"][0])
np.save(out_dir / "xdata2vec_middle3.npy", all_features["middle3"][0])
np.save(out_dir / "xdata2vec_acoustic.npy", all_features["acoustic"][0])
np.save(out_dir / "xdata2vec_stability.npy", all_features["stability"][0])
np.save(out_dir / "xdata2vec_fullspec.npy", all_features["fullspec"][0])
np.save(out_dir / "xdata2vec_all.npy", all_features["all"][0])

np.save(out_dir / "labels.npy", labels.astype(int))
np.save(out_dir / "file_list.npy", file_list.astype(object))
np.save(out_dir / "subject_ids.npy", subject_ids.astype(object))
np.save(out_dir / "label_map.npy", LABEL_MAP, allow_pickle=True)

try:
    (out_dir / "compute_log.json").write_text(json.dumps(EXTRACTION_STATS, indent=2))
except Exception as e:
    print(f"⚠️ Could not write compute_log.json: {e}")

config = {
    "dataset_root": str(DATASET_PATH),
    "classes": CLASS_DIRS,
    "label_map": LABEL_MAP,
    "subject_id_rule": "subject = filename prefix before '_' ; subject_id = '<class>/<subject>'",
    "model_name": model_name,
    "model_frozen": True,
    "seed": int(SEED),
    "env": EXTRACTION_ENV,
    "target_sr": int(TARGET_SR),
    "batch_size": int(BATCH_SIZE),
    "pooling": "masked_mean (attention-mask aware)",
    "layer_configs": LAYER_CONFIGS,
}
(out_dir / "extraction_config.json").write_text(json.dumps(config, indent=2))

print("\n✅ Done.")
print("Saved shapes:", {k: v[0].shape for k, v in all_features.items()})
print("Unique subjects saved:", int(np.unique(subject_ids).size))


# Russian 

In [ ]:
from __future__ import annotations

import json
import platform
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torchaudio
import transformers
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import Data2VecAudioModel

try:
    torchaudio.set_audio_backend("soundfile")
except Exception:
    pass
try:
    import librosa  # type: ignore
except Exception:
    librosa = None


# ==================== Step 0: environment ====================
EXTRACTION_ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": getattr(np, "__version__", "unknown"),
    "torch": getattr(torch, "__version__", "unknown"),
    "torchaudio": getattr(torchaudio, "__version__", "unknown"),
    "transformers": getattr(transformers, "__version__", "unknown"),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


# ==================== Step 1: Dataset traversal (2-class) ====================
REPO_ROOT = Path("/mnt/d/PhD/ALS_Diagnosis_Meta")
DATASET_PATH = REPO_ROOT / "Dataset" / "RUSSIAN"

CLASS_DIRS = ["CZ", "PZ"]  # stable mapping
LABEL_MAP = {"CZ": 0, "PZ": 1}

RE_FILE = re.compile(r"^(?P<prefix>[A-Za-z]{2})(?P<num>\d+)_")

def _canonical_class_from_prefix(prefix: str) -> str:
    p = prefix.upper()
    if p in {"CT", "CZ"}:
        return "CZ"
    if p == "PZ":
        return "PZ"
    raise ValueError(f"Unknown prefix '{prefix}' (expected CT/CZ/PZ)")

def _parse_subject_num(filename: str) -> tuple[str, str]:
    m = RE_FILE.match(filename)
    if not m:
        raise ValueError(f"Cannot parse subject from filename: {filename}")
    return m.group("prefix"), m.group("num")

@dataclass(frozen=True)
class Item:
    path: Path
    file_id: str
    subject_id: str
    label: int

if not DATASET_PATH.exists():
    raise RuntimeError(f"Missing dataset root: {DATASET_PATH}")

items: list[Item] = []
for p in sorted(DATASET_PATH.rglob("*.wav")):
    prefix, num = _parse_subject_num(p.name)
    cls = _canonical_class_from_prefix(prefix)
    file_id = str(p.relative_to(DATASET_PATH)).replace("\\", "/")
    subject_id = f"{cls}/{num}"
    items.append(Item(path=p, file_id=file_id, subject_id=subject_id, label=int(LABEL_MAP[cls])))

if not items:
    raise RuntimeError(f"No .wav files found under {DATASET_PATH}")

file_list = np.asarray([it.file_id for it in items], dtype=object)
subject_ids = np.asarray([it.subject_id for it in items], dtype=object)
labels = np.asarray([it.label for it in items], dtype=int)

print("\nSTEP 1) Dataset loaded")
print("  root:", DATASET_PATH)
print("  n_utterances:", int(len(items)))
print("  n_subjects:", int(np.unique(subject_ids).size))
print("  class_counts:", {k: int((labels == v).sum()) for k, v in LABEL_MAP.items()})


In [ ]:
# ==================== Step 2: Audio loading + resample to 16k ====================
TARGET_SR = 16000

class WavDataset(Dataset):
    def __init__(self, items: list[Item]):
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict:
        it = self.items[idx]
        try:
            wav, sr = torchaudio.load(str(it.path))  # (ch, n)
            if wav.ndim == 2 and wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)
            wav = wav.squeeze(0).contiguous()  # (n,)
        except Exception as e:
            if librosa is None:
                raise RuntimeError(
                    "Audio load failed via torchaudio and librosa is not available. "
                    "Install one of: torchcodec (for torchaudio default backend) or librosa/soundfile."
                ) from e
            wav_np, sr = librosa.load(str(it.path), sr=None, mono=True)
            wav = torch.tensor(wav_np, dtype=torch.float32).contiguous()

        if int(sr) != int(TARGET_SR):
            wav = torchaudio.transforms.Resample(orig_freq=int(sr), new_freq=int(TARGET_SR))(wav)

        return {"input_values": wav, "label": it.label, "subject_id": it.subject_id, "file_id": it.file_id}

def collate_fn(batch: list[dict]) -> dict:
    xs = [b["input_values"].float().squeeze() for b in batch]
    padded = pad_sequence(xs, batch_first=True, padding_value=0.0)  # (B, T)
    masks = [torch.ones(x.size(0), dtype=torch.long) for x in xs]
    padded_mask = pad_sequence(masks, batch_first=True, padding_value=0)  # (B, T)
    return {
        "input_values": padded,
        "attention_mask": padded_mask,
        "labels": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "subject_ids": [b["subject_id"] for b in batch],
        "file_ids": [b["file_id"] for b in batch],
    }

BATCH_SIZE = 1  # <-- if your system shuts off, keep this 1
loader = DataLoader(WavDataset(items), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print("\nSTEP 2) DataLoader ready | batch_size:", BATCH_SIZE, "| target_sr:", TARGET_SR)


In [ ]:
# ==================== Step 3: Load Data2Vec model (frozen) ====================
model_name = "facebook/data2vec-audio-large-960h"
print("\nSTEP 3) Loading model:", model_name)
model = Data2VecAudioModel.from_pretrained(model_name)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(device)

n_layers = int(getattr(model.config, "num_hidden_layers", 0))
if n_layers <= 0:
    raise ValueError("Could not determine model.config.num_hidden_layers for Data2Vec.")

print("  num_hidden_layers:", n_layers, "| hidden_size:", int(getattr(model.config, "hidden_size", -1)))


# ==================== Step 4: Layer configs (match Minsk naming) ====================
mid = n_layers // 2
stable_start = min(9, max(n_layers - 3, 0))
stable_layers = [stable_start, min(stable_start + 1, n_layers - 1), min(stable_start + 2, n_layers - 1)]
middle_layers = [max(mid - 1, 0), min(mid, n_layers - 1), min(mid + 1, n_layers - 1)]
multi_depth_layers = [0, stable_layers[0], middle_layers[-1], -1]

LAYER_CONFIGS: dict[str, list[int] | str] = {
    "stability": stable_layers,
    "acoustic": [0, 1, 2] if n_layers >= 3 else list(range(n_layers)),
    "middle3": middle_layers,
    "last3": [-3, -2, -1] if n_layers >= 3 else list(range(n_layers)),
    "fullspec": multi_depth_layers,
    "all": "ALL",
}

def _resolve_layers(spec: list[int] | str, *, n_states: int) -> list[int]:
    if spec == "ALL":
        return list(range(n_states))
    idxs: list[int] = []
    for x in spec:
        i = int(x)
        if i < 0:
            i = n_states + i
        if not (0 <= i < n_states):
            raise ValueError(f"Layer index out of range: {x} resolved to {i} with n_states={n_states}")
        idxs.append(i)
    return idxs


In [ ]:
# ==================== Step 5: Feature extraction (masked-mean pooling) ====================
print("\nSTEP 5) Extracting features (frozen forward only)...")
all_features: dict[str, list[np.ndarray]] = {name: [] for name in LAYER_CONFIGS.keys()}

batch_times_sec: list[float] = []
batch_sizes: list[int] = []
t_total_start = time.perf_counter()
total_utts = 0

for bi, batch in enumerate(loader, start=1):
    input_values = batch["input_values"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    bs = int(input_values.size(0))
    total_utts += bs

    t0 = time.perf_counter()
    with torch.inference_mode():
        outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    n_states = len(hidden_states)

    # compute pooled features for ALL layers once (faster / less overhead)
    pooled_by_layer = []
    for layer in range(n_states):
        feat = hidden_states[layer]  # (B, T', H)
        seq_len = feat.size(1)
        adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()
        masked = feat * adj_mask
        valid = adj_mask.sum(dim=1).clamp(min=1e-9)
        pooled_by_layer.append(masked.sum(dim=1) / valid)  # (B, H)

    for name, spec in LAYER_CONFIGS.items():
        layers = _resolve_layers(spec, n_states=n_states)
        final = torch.cat([pooled_by_layer[i] for i in layers], dim=-1).cpu().numpy()
        all_features[name].append(final)

    t1 = time.perf_counter()
    batch_times_sec.append(float(t1 - t0))
    batch_sizes.append(bs)

    if bi == 1 or bi % 10 == 0 or bi == int(np.ceil(len(items) / BATCH_SIZE)):
        print(f"  batch {bi:03d} | bs={bs} | time={batch_times_sec[-1]:.3f}s")

for name in list(all_features.keys()):
    all_features[name] = [np.concatenate(all_features[name], axis=0)]

t_total_end = time.perf_counter()
total_sec = float(t_total_end - t_total_start)

shapes = {k: v[0].shape for k, v in all_features.items()}
print("\nExtracted feature sets (shapes):", shapes)

EXTRACTION_STATS = {
    "total_utterances": int(total_utts),
    "batch_sizes": batch_sizes,
    "batch_times_sec": batch_times_sec,
    "total_time_sec": float(total_sec),
    "mean_time_sec_per_utt": float(total_sec / max(total_utts, 1)),
    "device": str(device),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}


In [ ]:
# ==================== Step 6: Save export ====================
out_dir = REPO_ROOT / "Embeddings" / "russian" / "raw" / "Data2vec"
out_dir.mkdir(parents=True, exist_ok=True)

print("\nSTEP 6) Saving to:", out_dir)

np.save(out_dir / "xdata2vec_last3.npy", all_features["last3"][0])
np.save(out_dir / "xdata2vec_middle3.npy", all_features["middle3"][0])
np.save(out_dir / "xdata2vec_acoustic.npy", all_features["acoustic"][0])
np.save(out_dir / "xdata2vec_stability.npy", all_features["stability"][0])
np.save(out_dir / "xdata2vec_fullspec.npy", all_features["fullspec"][0])
np.save(out_dir / "xdata2vec_all.npy", all_features["all"][0])

np.save(out_dir / "labels.npy", labels.astype(int))
np.save(out_dir / "file_list.npy", file_list.astype(object))
np.save(out_dir / "subject_ids.npy", subject_ids.astype(object))
np.save(out_dir / "label_map.npy", LABEL_MAP, allow_pickle=True)

(out_dir / "compute_log.json").write_text(json.dumps(EXTRACTION_STATS, indent=2))

config = {
    "dataset_root": str(DATASET_PATH),
    "classes": CLASS_DIRS,
    "label_map": LABEL_MAP,
    "subject_id_rule": "subject=num from filename prefix; subject_id = '<class>/<num>' (CT treated as CZ)",
    "model_name": model_name,
    "model_frozen": True,
    "env": EXTRACTION_ENV,
    "target_sr": int(TARGET_SR),
    "batch_size": int(BATCH_SIZE),
    "pooling": "masked_mean (attention-mask aware)",
    "layer_configs": LAYER_CONFIGS,
}
(out_dir / "extraction_config.json").write_text(json.dumps(config, indent=2))

print("\n✅ Done.")
print("Saved shapes:", {k: v[0].shape for k, v in all_features.items()})
print("Unique subjects saved:", int(np.unique(subject_ids).size))
